# 🔬 Agricultural Pest Lifelong Image Retrieval - Ablation Study on Kaggle (2 GPUs)

Notebook này được thiết kế để chạy thực nghiệm **6 module nghiên cứu loại trừ (Ablation Study)** cho tác vụ **Lifelong Image Retrieval** trên môi trường **Kaggle 2x T4 GPUs**.

### 6 cấu hình nghiên cứu loại trừ bao gồm:
1. **Base Retrieval**: Cấu hình cơ sở (`ip102_t1_retrieval.py`)
2. **All Attributes**: Sử dụng toàn bộ thuộc tính (`select_all_attr=True`, tắt các thành phần khác)
3. **All Attributes + OOD Gate**: Sử dụng toàn bộ thuộc tính kết hợp OOD Gate
4. **Attribute Selection**: Áp dụng bộ lọc thuộc tính (`Attribute Selection`)
5. **Similarity Restriction**: Áp dụng Ràng buộc độ tương đồng (`Similarity Restriction`)
6. **Known Uncertainty**: Cấu hình đầy đủ nhất có thêm `Known Uncertainty`

## 🛠️ Bước 1: Clone Repository & Submodules

In [1]:
import os
repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("-> Đang clone repository từ GitHub...")
    !git clone {repo_url} {working_dir}
else:
    print("-> Repository đã tồn tại. Đang tiến hành cập nhật (git pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# Tải mmyolo vào thư mục third_party nếu chưa có
if not os.path.exists("third_party/mmyolo"):
    print("-> Đang tải submodule mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("-> Submodule mmyolo đã có sẵn.")

-> Đang clone repository từ GitHub...
Cloning into '/kaggle/working/OW_OVD'...
remote: Enumerating objects: 1456, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 1456 (delta 32), reused 45 (delta 21), pack-reused 1391 (from 1)
Receiving objects: 100% (1456/1456), 3.16 MiB | 9.88 MiB/s, done.
Resolving deltas: 100% (996/996), done.
/kaggle/working/OW_OVD
-> Đang tải submodule mmyolo...
Cloning into 'third_party/mmyolo'...
remote: Enumerating objects: 4968, done.
remote: Counting objects: 100% (1355/1355), done.
remote: Compressing objects: 100% (290/290), done.
remote: Total 4968 (delta 1148), reused 1065 (delta 1065), pack-reused 3613 (from 1)
Receiving objects: 100% (4968/4968), 3.61 MiB | 12.39 MiB/s, done.
Resolving deltas: 100% (3217/3217), done.


## 📦 Bước 2: Cài đặt Dependencies & Vá lỗi MMCV

In [2]:
print("-> 1. Thiết lập phiên bản PyTorch & Torchvision...")
!pip install -q torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

print("\n-> 2. Cài đặt MMCV từ wheel index...")
!pip install -q mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

print("\n-> 3. Cài đặt các thư viện bổ trợ...")
!pip install -q matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

print("\n-> 4. Cài đặt MMDetection...")
!pip install -q "mmdet>=3.1.0" --no-deps

print("\n-> 5. Cài đặt MMYOLO từ source...")
!pip install -q --no-build-isolation --no-deps third_party/mmyolo

# Thiết lập HF Mirror để tăng tốc tải CLIP weights
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

print("\n-> 6. Vá lỗi kiểm tra phiên bản MMCV vật lý trên đĩa cứng...")
import site
import glob
import shutil

def patch_file(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        new_content = content
        for old_ver in ["'2.1.0'", "'2.2.0'", '"2.1.0"', '"2.2.0"']:
            new_content = new_content.replace(f"mmcv_maximum_version = {old_ver}", "mmcv_maximum_version = '2.3.0'")
        if new_content != content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            print(f"  [Vá lỗi] Đã cập nhật file: {file_path}")

def clear_pycache(root_dir):
    if not os.path.exists(root_dir):
        return
    for root, dirs, files in os.walk(root_dir):
        for d in dirs:
            if d == "__pycache__":
                pycache_path = os.path.join(root, d)
                try:
                    shutil.rmtree(pycache_path)
                except Exception:
                    pass

site_dirs = site.getsitepackages()
for s_dir in site_dirs:
    for pkg in ["mmdet", "mmyolo"]:
        pkg_dir = os.path.join(s_dir, pkg)
        patch_file(os.path.join(pkg_dir, "__init__.py"))
        clear_pycache(pkg_dir)

for init_file in glob.glob("**/mmyolo/__init__.py", recursive=True):
    patch_file(init_file)
    clear_pycache(os.path.dirname(init_file))
for init_file in glob.glob("**/mmdet/__init__.py", recursive=True):
    patch_file(init_file)
    clear_pycache(os.path.dirname(init_file))

paths_to_glob = [
    "/opt/conda/lib/python*/site-packages/mmdet/__init__.py",
    "/opt/conda/lib/python*/site-packages/mmyolo/__init__.py",
    "/usr/local/lib/python*/dist-packages/mmdet/__init__.py",
    "/usr/local/lib/python*/dist-packages/mmyolo/__init__.py"
]
for path_pattern in paths_to_glob:
    for init_file in glob.glob(path_pattern):
        patch_file(init_file)
        clear_pycache(os.path.dirname(init_file))

print("\n-> 7. Kiểm tra import tất cả các package...")
import torch
import mmcv

real_mmcv_version = mmcv.__version__
mmcv.__version__ = '2.0.1'

import mmdet
import mmyolo
mmcv.__version__ = real_mmcv_version

print(f"  - torch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"  - mmcv: {mmcv.__version__}")
print(f"  - mmdet: {mmdet.__version__}")
print(f"  - mmyolo: {mmyolo.__version__}")
print("====== Khởi tạo môi trường hoàn tất! ======")

-> 1. Thiết lập phiên bản PyTorch & Torchvision...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 83.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 81.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 121.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 81.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 48.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 69.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 77.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 71.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 65.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6

## 🗂️ Bước 3: Định vị Dataset & Sinh Đặc trưng nhãn bằng CLIP

In [3]:
import json
import torch
import numpy as np
import os
import glob
from transformers import AutoTokenizer, CLIPTextModelWithProjection

os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

# Kiểm tra model YOLO pretrain cục bộ trên Kaggle trước khi tải online
weights_path = 'pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
local_pretrain = '/kaggle/input/models/nhannguyen5578/yolo-world/pytorch/default/1/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'

if os.path.exists(local_pretrain):
    print(f"-> Phát hiện và sử dụng pretrain weights cục bộ từ: {local_pretrain}")
    if not os.path.exists(weights_path):
        import shutil
        shutil.copy(local_pretrain, weights_path)
else:
    if not os.path.exists(weights_path):
        print("-> Không tìm thấy pretrain weights cục bộ. Tiến hành tải online...")
        !wget -O {weights_path} https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth

dataset_root = None
for path in [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection',
    '/kaggle/input/ip102-for-object-detection',
    'data/IP102',
    '.'
]:
    if os.path.exists(os.path.join(path, 'train.json')):
        dataset_root = path
        break
if dataset_root is None:
    paths = glob.glob('/kaggle/input/**/train.json', recursive=True)
    if paths:
        dataset_root = os.path.dirname(paths[0])

print(f"-> Thư mục Dataset IP102: {dataset_root}")
class_names = [str(i) for i in range(102)]

class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

print("-> Đang sinh đặc trưng nhãn bằng CLIP...")
model_name = 'openai/clip-vit-base-patch32'
local_clip_path = "/kaggle/input/models/yujkaggle/openaiclip-vit-base-patch32/pytorch/default/1"
working_clip_path = "/kaggle/working/openaiclip-vit-base-patch32"

if os.path.exists(local_clip_path):
    print(f"-> Phát hiện mô hình CLIP offline từ: {local_clip_path}")
    # Để tránh lỗi ValueError: Due to a serious vulnerability issue in torch.load (CVE-2025-32434) trên PyTorch < 2.6,
    # chúng ta tự động chuyển đổi file .bin sang tệp định dạng an toàn .safetensors ở thư mục working.
    if not os.path.exists(os.path.join(working_clip_path, "model.safetensors")):
        print("-> Đang tiến hành chuyển đổi từ pytorch_model.bin sang model.safetensors...")
        import shutil
        from safetensors.torch import save_file
        os.makedirs(working_clip_path, exist_ok=True)
        for fname in os.listdir(local_clip_path):
            if fname != "pytorch_model.bin":
                shutil.copy(os.path.join(local_clip_path, fname), os.path.join(working_clip_path, fname))
        
        # Load bằng PyTorch CPU, ép kiểu liền mạch (contiguous) cho các tensor không liền nhau và ghi lại dưới dạng safetensors
        state_dict = torch.load(os.path.join(local_clip_path, "pytorch_model.bin"), map_location="cpu")
        state_dict = {k: v.contiguous() if isinstance(v, torch.Tensor) else v for k, v in state_dict.items()}
        save_file(state_dict, os.path.join(working_clip_path, "model.safetensors"))
        print(f"-> Chuyển đổi thành công! Đã lưu tại: {working_clip_path}")
    model_name = working_clip_path

tokenizer = AutoTokenizer.from_pretrained(model_name)
clip_model = CLIPTextModelWithProjection.from_pretrained(model_name, use_safetensors=True)
clip_model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = clip_model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))

num_att = len(class_names) * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')

thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')
print("====== Khởi tạo và sinh đặc trưng nhãn hoàn tất! ======")

-> Không tìm thấy pretrain weights cục bộ. Tiến hành tải online...
--2026-08-29 14:24:57--  https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
Resolving huggingface.co (huggingface.co)... 13.226.251.20, 13.226.251.112, 13.226.251.81, ...
Connecting to huggingface.co (huggingface.co)|13.226.251.20|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65bb7a71626a4c209906adf5/09dafb73b0d19d270cf20f7eeac6a7861303a753332d5df9917772ba23e4a47d?user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth%3B+filename%3D%22yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth%22%3B&Expires=1788017097&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjViYjdhNzE2MjZhNGMyMDk5MDZhZGY1LzA5ZGFmYjczYjBkMTlkMjcwY2YyMGY3ZWVhYzZhNzg2MTMwM2E3NTMzMzJ

/tmp/ipykernel_23/3661944515.py:67: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(os.path.join(local_clip_path, "pytorch_model.bin"), map_location="c

-> Chuyển đổi thành công! Đã lưu tại: /kaggle/working/openaiclip-vit-base-patch32


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: /kaggle/working/openaiclip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_m

====== Khởi tạo và sinh đặc trưng nhãn hoàn tất! ======


## 📂 Bước 4: Tạo các File Cấu hình (Ablation Configs)
Ghi mới 6 file cấu hình ablation vào thư mục `/kaggle/working/configs/retrieval_abl`.

In [4]:
import os
os.makedirs("configs/retrieval_abl", exist_ok=True)
print("-> Đã tạo thư mục configs/retrieval_abl")

-> Đã tạo thư mục configs/retrieval_abl


In [5]:
%%writefile configs/retrieval_abl/ip102_t1_retrieval.py
_base_ = '../../NewRetrieval_02/ip102_t1_retrieval.py'


Overwriting configs/retrieval_abl/ip102_t1_retrieval.py


In [6]:
%%writefile configs/retrieval_abl/ip102_t1_retrieval_all_attr.py
_base_ = '../../NewRetrieval_02/ip102_t1_retrieval.py'

model = dict(
    bbox_head=dict(
        select_all_attr=True,
        use_top_k_att=False,
        use_ood_gate=False,
        use_known_uncertainty=False
    )
)


Writing configs/retrieval_abl/ip102_t1_retrieval_all_attr.py


In [7]:
%%writefile configs/retrieval_abl/ip102_t1_retrieval_all_attr_ood.py
_base_ = '../../NewRetrieval_02/ip102_t1_retrieval.py'

model = dict(
    bbox_head=dict(
        select_all_attr=True,
        use_top_k_att=False,
        use_ood_gate=True,
        use_ood_prob=True,
        use_known_uncertainty=False
    )
)


Writing configs/retrieval_abl/ip102_t1_retrieval_all_attr_ood.py


In [8]:
%%writefile configs/retrieval_abl/ip102_t1_retrieval_attr_sel.py
_base_ = './ip102_t1_retrieval_all_attr_ood.py'

model = dict(
    bbox_head=dict(
        select_all_attr=False,
        selected_att_path='data/IP102/selected_att_embeddings.pth',
        attr_sel_for_known_only=False,
        use_top_k_att=False,
        use_ood_gate=True,
        use_ood_prob=False,
        use_known_uncertainty=False
    )
)


Writing configs/retrieval_abl/ip102_t1_retrieval_attr_sel.py


In [9]:
%%writefile configs/retrieval_abl/ip102_t1_retrieval_sim_restr.py
_base_ = './ip102_t1_retrieval_attr_sel.py'

model = dict(
    bbox_head=dict(
        use_similarity_restriction=True,
        sim_restr_beta=0.2,
        selected_att_path='data/IP102/selected_att_embeddings_sim_restr.pth',
        use_known_uncertainty=False
    )
)


Overwriting configs/retrieval_abl/ip102_t1_retrieval_sim_restr.py


In [10]:
%%writefile configs/retrieval_abl/ip102_t1_retrieval_known_uncer.py
_base_ = './ip102_t1_retrieval_sim_restr.py'

model = dict(
    bbox_head=dict(
        use_known_uncertainty=True
    )
)


Writing configs/retrieval_abl/ip102_t1_retrieval_known_uncer.py


## 🚀 Bước 5: Huấn luyện Phân tán trên 2 GPUs
Chạy huấn luyện tuần tự 6 module cấu hình sử dụng `torchrun` với cấu hình 2 GPUs song song để tối ưu tốc độ học.

In [11]:
import subprocess
import os

configs = [
    "configs/retrieval_abl/ip102_t1_retrieval.py",
    "configs/retrieval_abl/ip102_t1_retrieval_all_attr.py"
    # "configs/retrieval_abl/ip102_t1_retrieval_all_attr_ood.py",
    # "configs/retrieval_abl/ip102_t1_retrieval_attr_sel.py"
    # "configs/retrieval_abl/ip102_t1_retrieval_sim_restr.py",
    # "configs/retrieval_abl/ip102_t1_retrieval_known_uncer.py"
]

for cfg in configs:
    name = os.path.splitext(os.path.basename(cfg))[0]
    work_dir = f"work_dirs/{name}"
    print("="*80)
    print(f"🔥 Bắt đầu huấn luyện phân tán 2 GPUs cho: {name}")
    print("="*80)
    
    cmd = [
        "python", "-m", "torch.distributed.run",
        "--nproc_per_node=2",
        "--master_port", "29550",
        "third_party/mmyolo/tools/train.py",
        cfg,
        "--launcher", "pytorch",
        "--work-dir", work_dir
    ]
    
    # Thiết lập PYTHONPATH=. và HF_ENDPOINT cho môi trường huấn luyện
    env = os.environ.copy()
    env["PYTHONPATH"] = "."
    env["HF_ENDPOINT"] = "https://hf-mirror.com"
    
    subprocess.run(cmd, env=env, check=True)
    print(f"✅ Hoàn tất huấn luyện: {name}\n")

🔥 Bắt đầu huấn luyện phân tán 2 GPUs cho: ip102_t1_retrieval


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/dist-packages/mmdet/models/backbones/trident_resnet.py:2

08/29 14:26:34 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
08/29 14:26:34 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
08/29 14:26:35 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
    CUDA available: True
    MUSA available: False

/kaggle/working/OW_OVD/yolo_world/models/dense_heads/our_head_new.py:357: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  atts = torch.load(att_embeddings)
/kaggle/working/OW_

08/29 14:26:36 - mmengine - INFO - Using SyncBatchNorm()
08/29 14:26:36 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
(LOWEST      ) EarlyStoppingHook                  
 -------------------- 
before_train:
(9           ) YOLOv5ParamSchedulerHook           
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
(NORMAL      ) PipelineSwitchHook                 
(NORMAL      ) OurWorkPiplineHook                 
 -------------------- 
before_train_iter:
(9           ) YOLOv5ParamSchedulerHook           
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      

...

[OurHeadRetrieval] bbox_head.text_projection.weight not found in state_dict. Initializing text_projection randomly.
[OurHead] Dynamically resizing self.att_embeddings from torch.Size([2550, 512]) to torch.Size([175, 512]) to match checkpoint.
[OurHeadRetrieval] bbox_head.text_projection.weight not found in state_dict. Initializing text_projection randomly.
[OurHead] Dynamically resizing self.att_embeddings from torch.Size([2550, 512]) to torch.Size([175, 512]) to match checkpoint.
The model and loaded state dict do not match exactly

size mismatch for embeddings: copying a param with shape torch.Size([25, 512]) from checkpoint, the shape in current model is torch.Size([102, 512]).
missing keys in source state_dict: bbox_head.head_module.ret_preds.0.0.conv.weight, bbox_head.head_module.ret_preds.0.0.bn.weight, bbox_head.head_module.ret_preds.0.0.bn.bias, bbox_head.head_module.ret_preds.0.0.bn.running_mean, bbox_head.head_module.ret_preds.0.0.bn.running_var, bbox_head.head_module.ret_pre

/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()
/usr/local/lib/python3.12/dist-packages/albumentations/core/composition.py:331: UserWarning: Got processor for bboxes, but no transform to process it.
  self._set_keys()
/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/29 14:35:08 - mmengine - INFO - Exp name: ip102_t1_retrieval_all_attr_20260829_143237
08/29 14:35:08 - mmengine - INFO - Epoch(train) [1][48/48]  base_lr: 1.0000e-04 lr: 4.7000e-06  eta: 0:00:00  time: 3.0725  data_time: 0.0692  memory: 13896  grad_norm: nan  loss: 287.1810  loss_cls: 134.2713  loss_bbox: 63.7797  loss_dfl: 88.2998  loss_retrieval: 0.8301  loss_dwopp: 0.0000
Using all attributes (select_all_attr=True), total: 2550.
disable log
Using all attributes (select_all_attr=True), total: 2550.
disable log
08/29 14:35:08 - mmengine - INFO - Saving checkpoint at 1 epochs
08/29 14:35:08 - mmengine - WARNING - `save_param_scheduler` is True but `self.param_schedulers` is None, so skip saving parameter schedulers
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=Tr

[rank0]:[W829 14:37:59.623867222 ProcessGroupNCCL.cpp:1168] Warning: WARNING: process group has NOT been destroyed before we destruct ProcessGroupNCCL. On normal program exit, the application should call destroy_process_group to ensure that any pending NCCL operations have finished in this process. In rare cases this process can exit before this point and block the progress of another member of the process group. This constraint has always been present,  but this warning has only been added since PyTorch 2.4 (function operator())


✅ Hoàn tất huấn luyện: ip102_t1_retrieval_all_attr



## 📊 Bước 6: Chạy đánh giá (Retrieval & Open-World Metrics)
Tiến hành đánh giá mô hình bằng script `evaluate_retrieval.py` cho cả 6 module.

In [12]:
import os
import subprocess

# Tự động phát hiện đường dẫn dataset trên Kaggle
if os.path.exists("/kaggle/input/datasets/nta212/ip102-for-object-detection"):
    dataset_root = "/kaggle/input/datasets/nta212/ip102-for-object-detection"
elif os.path.exists("IP102 dataset"):
    dataset_root = "IP102 dataset"
else:
    dataset_root = "data/IP102"

print(f"Using dataset root: {dataset_root}")

configs = [
    "configs/retrieval_abl/ip102_t1_retrieval.py",
    "configs/retrieval_abl/ip102_t1_retrieval_all_attr.py"
    # "configs/retrieval_abl/ip102_t1_retrieval_all_attr_ood.py",
    # "configs/retrieval_abl/ip102_t1_retrieval_attr_sel.py"
    # "configs/retrieval_abl/ip102_t1_retrieval_sim_restr.py",
    # "configs/retrieval_abl/ip102_t1_retrieval_known_uncer.py"
]

# Phát hiện model CLIP offline hoặc thư mục đã được convert safetensors cục bộ
# local_clip_path = "/kaggle/input/models/yujkaggle/openaiclip-vit-base-patch32/pytorch/default/1"
# working_clip_path = "/kaggle/working/openaiclip-vit-base-patch32"
# --- CẤU HÌNH KIỂU TRÍCH XUẤT ĐẶC TRƯNG ---
extractor_type = "vit"  # Hoặc "dinov2"
extractor_model = "/kaggle/input/models/oleksandrkharytonov/googlevit-base-patch16-224/pytorch/default/1"  # Đường dẫn ViT trên Kaggle của anh

use_clip_path = "openai/clip-vit-base-patch32"
if os.path.exists(working_clip_path):
    use_clip_path = working_clip_path
    print(f"-> Sử dụng mô hình CLIP offline (sau khi đã convert sang safetensors) tại: {use_clip_path}")
elif os.path.exists(local_clip_path):
    use_clip_path = local_clip_path
    print(f"-> Cảnh báo: Sử dụng mô hình CLIP offline gốc tại: {use_clip_path}")

for cfg in configs:
    name = os.path.splitext(os.path.basename(cfg))[0]
    ckpt_path = f"work_dirs/{name}/best_coco_Current class AP50_epoch_5.pth"
    if not os.path.exists(ckpt_path):
        ckpt_path = f"work_dirs/{name}/epoch_5.pth"
        
    if not os.path.exists(ckpt_path):
        print(f"⚠️ Không tìm thấy checkpoint cho {name} tại {ckpt_path}. Bỏ qua.")
        continue
        
    print(f"\n---> Đang đánh giá {name} (Checkpoint: {ckpt_path}) <---")
    
    # Thiết lập PYTHONPATH=. và HF_ENDPOINT cho môi trường đánh giá
    env = os.environ.copy()
    env["PYTHONPATH"] = "."
    env["HF_ENDPOINT"] = "https://hf-mirror.com"
    
    # 1. Đánh giá thông thường dùng CLIP Feature
    print(f"[CLIP Feature Extraction]")
    cmd_clip = [
        "python", "evaluate_retrieval.py",
        "--config", cfg,
        "--checkpoint", ckpt_path,
        "--dataset-root", dataset_root,
        "--query-split", "test",
        "--gallery-split", "val",
        "--clip-model", use_clip_path,
        "--query-cache", f"work_dirs/{name}/query_cache_clip.pkl",
        "--gallery-cache", f"work_dirs/{name}/gallery_cache_clip.pkl",
        "--output-report", f"work_dirs/{name}/report_clip.md"
    ]
    subprocess.run(cmd_clip, env=env, check=True)

    # 2. Đánh giá sử dụng Retrieval Head của mô hình tự học
    print(f"[Detector Learned Feature Extraction]")
    cmd_det = [
        "python", "evaluate_retrieval.py",
        "--config", cfg,
        "--extractor-type", extractor_type,       # <--- THÊM DÒNG NÀY
        "--extractor-model", extractor_model,
        "--checkpoint", ckpt_path,
        "--dataset-root", dataset_root,
        "--query-split", "test",
        "--gallery-split", "val",
        "--detector-retrieval",
        # "--query-cache", f"work_dirs/{name}/query_cache_det.pkl",
        "--query-cache", f"work_dirs/{name}/query_cache_{extractor_type}.pkl",
        "--gallery-cache", f"work_dirs/{name}/gallery_cache_{extractor_type}.pkl",
        "--output-report", f"work_dirs/{name}/report_{extractor_type}.md"
        # "--gallery-cache", f"work_dirs/{name}/gallery_cache_det.pkl",
        # "--output-report", f"work_dirs/{name}/report_detector.md"
    ]
    subprocess.run(cmd_det, env=env, check=True)

Using dataset root: /kaggle/input/datasets/nta212/ip102-for-object-detection
-> Sử dụng mô hình CLIP offline (sau khi đã convert sang safetensors) tại: /kaggle/working/openaiclip-vit-base-patch32

---> Đang đánh giá ip102_t1_retrieval (Checkpoint: work_dirs/ip102_t1_retrieval/best_coco_Current class AP50_epoch_1.pth) <---
[CLIP Feature Extraction]


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/kaggle/working/OW_OVD/yolo_world/models/dense_heads/our_head_new.py:357: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by th

-> Detected offline Kaggle CLIP model. Defaulting to: /kaggle/input/models/yujkaggle/openaiclip-vit-base-patch32/pytorch/default/1
      IP102 RETRIEVAL METRICS EVALUATION PIPELINE      
-> Reading annotations...
-> WARNING: Standard subfolders empty. Scanning /kaggle/input/datasets/nta212/ip102-for-object-detection recursively to locate image files...
-> Auto-resolved image directory to: /kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/VOC2007/JPEGImages
-> WARNING: Standard subfolders empty. Scanning /kaggle/input/datasets/nta212/ip102-for-object-detection recursively to locate image files...
-> Auto-resolved image directory to: /kaggle/input/datasets/nta212/ip102-for-object-detection/VOC2007/VOC2007/JPEGImages
-> Found 2713 query images and 2176 gallery images.
-> Query cache not found or incomplete. Extracting embeddings on the fly...
Loads checkpoint by local backend from path: work_dirs/ip102_t1_retrieval/best_coco_Current class AP50_epoch_1.pth
[OurHead] Dynamica

Processing Query Split (BBox Detection):   0%|          | 0/2713 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Processing Gallery Split (BBox Detection):   0%|          | 0/2176 [00:00<?, ?it/s]

-> Saved Query cache to work_dirs/ip102_t1_retrieval/query_cache_clip.pkl
-> Gallery cache not found. Extracting embeddings on the fly...


...


Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all att

Matching Queries:  30%|███       | 817/2713 [00:00<00:00, 8166.31it/s]


Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all attributes (select_all_attr=True), total: 2550.
Using all att

Matching Queries: 100%|██████████| 2713/2713 [00:00<00:00, 9403.30it/s]


-> Calculating Open-World anomaly detection metrics...
-> Task class configuration: prev_intro_cls=0, cur_intro_cls=7 (Total Known: 7)
-> Query set OOD breakdown: 0 Known samples, 2713 Unknown (unseen) samples.
-> Skipped AUROC calculation because either Known or Unknown query samples are missing.

            SUMMARY RETRIEVAL METRICS            
Recall@1:  0.1047
Recall@5:  0.2714
Recall@10: 0.4160
-> Saved markdown evaluation report to: work_dirs/ip102_t1_retrieval_all_attr/report_vit.md
====== Evaluation Process Completed Successfully! ======


## 📈 Bước 7: Tổng Hợp và Hiển Thị Kết Quả So Sánh
Tự động đọc các file report vừa tạo và in ra bảng tổng hợp kết quả so sánh giữa các cấu hình nghiên cứu loại trừ.

In [13]:
import re
import os

configs = [
    "ip102_t1_retrieval",
    "ip102_t1_retrieval_all_attr"
    # "ip102_t1_retrieval_all_attr_ood",
    # "ip102_t1_retrieval_attr_sel"
    # "ip102_t1_retrieval_sim_restr",
    # "ip102_t1_retrieval_known_uncer"
]

def extract_metrics(report_path):
    if not os.path.exists(report_path):
        return "-", "-", "-", "-", "-"
    with open(report_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    # Trích xuất R@1, R@5, R@10 từ bảng Summary Metrics
    r1 = re.search(r"Recall@1\s*\|\s*([0-9.]+)", content)
    r5 = re.search(r"Recall@5\s*\|\s*([0-9.]+)", content)
    r10 = re.search(r"Recall@10\s*\|\s*([0-9.]+)", content)
    
    # Trích xuất AUROC và FPR@TPR95
    auroc = re.search(r"AUROC \(Area Under ROC\)\s*\|\s*([0-9.]+)", content)
    fpr95 = re.search(r"FPR@TPR95\s*\|\s*([0-9.]+)", content)
    
    return (
        r1.group(1) if r1 else "-",
        r5.group(1) if r5 else "-",
        r10.group(1) if r10 else "-",
        auroc.group(1) if auroc else "-",
        fpr95.group(1) if fpr95 else "-"
    )

print("# BẢNG TỔNG HỢP KẾT QUẢ NGHIÊN CỨU LOẠI TRỪ (ABLATION STUDY)\n")
print("## 1. Kết quả sử dụng Detector Retrieval Head (Learned Features)\n")
print("| Configuration | Recall@1 | Recall@5 | Recall@10 | AUROC | FPR@TPR95 |")
print("| :--- | :---: | :---: | :---: | :---: | :---: |")
for cfg in configs:
    report = f"work_dirs/{cfg}/report_vit.md"
    r1, r5, r10, auc, fpr = extract_metrics(report)
    print(f"| {cfg} | {r1} | {r5} | {r10} | {auc} | {fpr} |")
    
print(f"\n## 2. Kết quả sử dụng {extractor_type.upper()} Crop-then-Search ({extractor_type.upper()} Features)\n")
print("| Configuration | Recall@1 | Recall@5 | Recall@10 | AUROC | FPR@TPR95 |")
print("| :--- | :---: | :---: | :---: | :---: | :---: |")
for cfg in configs:
    report = f"work_dirs/{cfg}/report_{extractor_type}.md"   # <--- ĐỔI report_clip.md thành report_{extractor_type}.md
    r1, r5, r10, auc, fpr = extract_metrics(report)
    print(f"| {cfg} | {r1} | {r5} | {r10} | {auc} | {fpr} |")

# print("\n## 2. Kết quả sử dụng CLIP Crop-then-Search (CLIP Features)\n")
# print("| Configuration | Recall@1 | Recall@5 | Recall@10 | AUROC | FPR@TPR95 |")
# print("| :--- | :---: | :---: | :---: | :---: | :---: |")
# for cfg in configs:
#     report = f"work_dirs/{cfg}/report_clip.md"
#     r1, r5, r10, auc, fpr = extract_metrics(report)
#     print(f"| {cfg} | {r1} | {r5} | {r10} | {auc} | {fpr} |")

# BẢNG TỔNG HỢP KẾT QUẢ NGHIÊN CỨU LOẠI TRỪ (ABLATION STUDY)

## 1. Kết quả sử dụng Detector Retrieval Head (Learned Features)

| Configuration | Recall@1 | Recall@5 | Recall@10 | AUROC | FPR@TPR95 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| ip102_t1_retrieval | - | - | - | - | - |
| ip102_t1_retrieval_all_attr | - | - | - | - | - |

## 2. Kết quả sử dụng VIT Crop-then-Search (VIT Features)

| Configuration | Recall@1 | Recall@5 | Recall@10 | AUROC | FPR@TPR95 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| ip102_t1_retrieval | - | - | - | - | - |
| ip102_t1_retrieval_all_attr | - | - | - | - | - |
